## Hotel Management AI Agent

In [27]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio

In [28]:
load_dotenv(override=True)
MODEL = "gpt-4o-mini"
openai_api_key = os.getenv("OPENAI_API_KEY")
openai = OpenAI(api_key=openai_api_key)

In [29]:
system_prompt = """
You are a helpful assistant for a small hotel front desk.
Your job is to manage customer records only - adding, viewing, updating and deleting them.

Replies must be polite, short and professional.
Ask for missing details in small steps, one or two at a time. Never ask for everything at once.
Never guess or invent missing details. Always ask the staff member for them.
Repeat back what was saved, changed or deleted so staff can check it.
Show only the customer records the staff member asked for.
Only confirm an action was done after it has really succeeded.
Explain any problems in friendly, plain language.

For requests unrelated to customer records - such as payments, pricing or anything else - politely explain that you can only help with customer records.

Example conversation — adding a customer:
Staff: Register a new guest, Jane Smith, room 204.
Agent: Happy to help. What is Jane's phone number, and her check-in and check-out dates?
Staff: 0712345678, from 2026-10-01 to 2026-10-04.
Agent: Saved! Customer #1: Jane Smith, room 204, 1 to 4 October 2026.

Example conversation — deleting a customer:
Staff: Delete customer 1.
Agent: Do you really want to delete Jane Smith (room 204)? Please reply yes or no.
Staff: yes
Agent: Done. Customer #1 has been deleted.
"""

In [30]:
room_numbers = {
    "100":"empty",
    "101":"empty",
    "102":"empty",
    "103":"empty",
    "104":"empty",
    "105":"empty",
    "106":"empty",
    "107":"empty",
    "108":"empty",
    "109":"empty",
    "110":"empty",
    "111":"empty"
}

In [31]:
def initialize_rooms():
    print("Initializing rooms", flush=True)
    with sqlite3.connect(DB) as conn:
        conn.execute("PRAGMA foreign_keys = ON")
        cursor = conn.cursor()

        for room_number, status in room_numbers.items():
            cursor.execute('''
                INSERT OR IGNORE INTO rooms (room_number, is_available)
                VALUES (?, ?)
            ''', (room_number, 1 if status == "empty" else 0))

        conn.commit()
        return f"{len(room_numbers)} rooms initialized successfully."

## Database

In [32]:
import sqlite3

In [33]:
DB = "hotel.db"

with sqlite3.connect(DB) as conn:
    conn.execute("PRAGMA foreign_keys = ON")
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS rooms (
            room_number   TEXT     PRIMARY KEY,
            room_type     TEXT,
            is_available  INTEGER  DEFAULT 1
        )
    ''')
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS customers (
            customer_id      INTEGER  PRIMARY KEY AUTOINCREMENT,
            name             TEXT     NOT NULL CHECK(TRIM(name) != ''),
            phone            TEXT     NOT NULL CHECK(
                                         LENGTH(REPLACE(phone, '+', '')) BETWEEN 9 AND 15
                                     ),
            email            TEXT     CHECK(
                                         email IS NULL OR email LIKE '%_@_%.__%'
                                     ),
            room_number      TEXT     NOT NULL,
            check_in_date    TEXT     NOT NULL CHECK(
                                         check_in_date GLOB '[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]'
                                     ),
            check_out_date   TEXT     NOT NULL CHECK(
                                         check_out_date GLOB '[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]'
                                         AND check_out_date > check_in_date
                                     ),
            special_requests TEXT,
            FOREIGN KEY (room_number) REFERENCES rooms(room_number)
        )
    ''')

    conn.commit()

initialize_rooms()

Initializing rooms


'12 rooms initialized successfully.'

## Tools

In [34]:
def get_customer_details(identifier):
    print(f"DATABASE TOOL CALLED: Getting customer details for {identifier}", flush=True)
    with sqlite3.connect(DB) as conn:
        conn.execute("PRAGMA foreign_keys = ON")
        cursor = conn.cursor()

        if str(identifier).isdigit():
            cursor.execute('SELECT * FROM customers WHERE customer_id = ?', (int(identifier),))
            result = cursor.fetchone()

            if not result:
                return f"No customer found with ID {identifier}."
            
            return (
                f"Customer Found:\n"
                f"ID: {result[0]}\n"
                f"Name: {result[1]}\n"
                f"Phone: {result[2]}\n"
                f"Email: {result[3]}\n"
                f"Room: {result[4]}\n"
                f"Check-in: {result[5]}\n"
                f"Check-out: {result[6]}\n"
                f"Special Requests: {result[7] or 'None'}"
            )

        else:
            cursor.execute('SELECT * FROM customers WHERE name = ?', (str(identifier).lower(),))
            results = cursor.fetchall()

            if not results:
                return f"No customer found with the name '{identifier}'."

            if len(results) == 1:
                r = results[0]
                return (
                    f"Customer Found:\n"
                    f"ID: {r[0]}\n"
                    f"Name: {r[1]}\n"
                    f"Phone: {r[2]}\n"
                    f"Email: {r[3]}\n"
                    f"Room: {r[4]}\n"
                    f"Check-in: {r[5]}\n"
                    f"Check-out: {r[6]}\n"
                    f"Special Requests: {r[7] or 'None'}"
                )

            matches = "\n".join(
                f"  ID: {r[0]} | Room: {r[4]} | Check-in: {r[5]}"
                for r in results
            )
            return (
                f"Multiple customers found with the name '{identifier}':\n"
                f"{matches}\n"
                f"Please provide the customer ID to get the exact record."
            )

In [35]:
def save_customer_details(name, phone, email, room_number, check_in_date, check_out_date, special_requests=None):
    print(f"DATABASE TOOL CALLED: Saving customer details for {name}", flush=True)
    with sqlite3.connect(DB) as conn:
        conn.execute("PRAGMA foreign_keys = ON")
        cursor = conn.cursor()

        cursor.execute('SELECT * FROM rooms WHERE room_number = ?', (room_number,))
        room = cursor.fetchone()
        if not room:
            return f"Room {room_number} does not exist"

        cursor.execute('SELECT * FROM rooms WHERE room_number = ? AND is_available = 1', (room_number,))
        available = cursor.fetchone()
        if not available:
            return f"Room {room_number} is not available"

        cursor.execute('''
            INSERT INTO customers (name, phone, email, room_number, check_in_date, check_out_date, special_requests)
            VALUES (?, ?, ?, ?, ?, ?, ?)
        ''', (name.lower(), phone, email, room_number, check_in_date, check_out_date, special_requests))

        cursor.execute('UPDATE rooms SET is_available = 0 WHERE room_number = ?', (room_number,))

        conn.commit()
        return f"Customer {name} saved successfully in room {room_number}"

In [36]:
UNCHANGED = object()

def update_customer_details(customer_id, name=UNCHANGED, phone=UNCHANGED, email=UNCHANGED,
                             room_number=UNCHANGED, check_in_date=UNCHANGED,
                             check_out_date=UNCHANGED, special_requests=UNCHANGED):
    print(f"DATABASE TOOL CALLED: Updating customer details for ID {customer_id}", flush=True)
    with sqlite3.connect(DB) as conn:
        conn.execute("PRAGMA foreign_keys = ON")
        cursor = conn.cursor()

        cursor.execute('SELECT * FROM customers WHERE customer_id = ?', (int(customer_id),))
        existing = cursor.fetchone()
        if not existing:
            return f"No customer found with ID {customer_id}."

        if room_number is not UNCHANGED and room_number != existing[4]:
            cursor.execute('SELECT * FROM rooms WHERE room_number = ?', (room_number,))
            room = cursor.fetchone()
            if not room:
                return f"Room {room_number} does not exist."

            cursor.execute('SELECT * FROM rooms WHERE room_number = ? AND is_available = 1', (room_number,))
            available = cursor.fetchone()
            if not available:
                return f"Room {room_number} is not available."

            cursor.execute('UPDATE rooms SET is_available = 1 WHERE room_number = ?', (existing[4],))
            cursor.execute('UPDATE rooms SET is_available = 0 WHERE room_number = ?', (room_number,))

        fields = {
            "name"             : name.lower() if name is not UNCHANGED and name else name,
            "phone"            : phone,
            "email"            : email,
            "room_number"      : room_number,
            "check_in_date"    : check_in_date,
            "check_out_date"   : check_out_date,
            "special_requests" : special_requests
        }

        updates = {k: v for k, v in fields.items() if v is not UNCHANGED}

        if not updates:
            return "No fields provided to update."

        set_clause = ", ".join(f"{k} = ?" for k in updates.keys())
        values     = list(updates.values()) + [int(customer_id)]
        cursor.execute(f"UPDATE customers SET {set_clause} WHERE customer_id = ?", values)

        conn.commit()

        field_index = {
            "name": 1, "phone": 2, "email": 3,
            "room_number": 4, "check_in_date": 5,
            "check_out_date": 6, "special_requests": 7
        }
        changed = "\n".join(
            f"  {k}: {existing[field_index[k]]} to {v}"
            for k, v in updates.items()
        )
        return f"Customer ID {customer_id} updated successfully:\n{changed}"

In [37]:
def delete_customer(customer_id, confirmation=None):
    print(f"DATABASE TOOL CALLED: Delete customer ID {customer_id}", flush=True)
    with sqlite3.connect(DB) as conn:
        conn.execute("PRAGMA foreign_keys = ON")
        cursor = conn.cursor()

        cursor.execute('SELECT * FROM customers WHERE customer_id = ?', (int(customer_id),))
        existing = cursor.fetchone()
        if not existing:
            return f"No customer found with ID {customer_id}."

        if confirmation is None:
            return (
                f"Are you sure you want to delete this customer?\n"
                f"  ID: {existing[0]}\n"
                f"  Name: {existing[1]}\n"
                f"  Room: {existing[4]}\n"
                f"  Check-in: {existing[5]}\n"
                f"  Check-out: {existing[6]}\n"
                f"Reply YES to confirm or NO to cancel."
            )

        if confirmation.strip().lower() != "yes":
            return "Deletion cancelled. No changes were made."

        cursor.execute('UPDATE rooms SET is_available = 1 WHERE room_number = ?', (existing[4],))

        cursor.execute('DELETE FROM customers WHERE customer_id = ?', (int(customer_id),))

        conn.commit()
        return f"Customer ID {customer_id} ({existing[1]}) deleted successfully. Room {existing[4]} is now available."

In [38]:
def check_rooms():
    print("DATABASE TOOL CALLED: Checking room availability", flush=True)
    with sqlite3.connect(DB) as conn:
        conn.execute("PRAGMA foreign_keys = ON")
        cursor = conn.cursor()

        cursor.execute('SELECT room_number, is_available FROM rooms')
        results = cursor.fetchall()

        if not results:
            return "No rooms found in the database."

        available  = [r[0] for r in results if r[1] == 1]
        occupied   = [r[0] for r in results if r[1] == 0]

        return (
            f"Total Rooms: {len(results)}\n"
            f"Available ({len(available)}): {', '.join(available) if available else 'None'}\n"
            f"Occupied  ({len(occupied)}):  {', '.join(occupied)  if occupied  else 'None'}"
        )

In [39]:
def list_all_customers():
    print("DATABASE TOOL CALLED: Listing all customers", flush=True)
    with sqlite3.connect(DB) as conn:
        conn.execute("PRAGMA foreign_keys = ON")
        cursor = conn.cursor()

        cursor.execute('SELECT * FROM customers')
        results = cursor.fetchall()

        if not results:
            return "No customers registered yet."

        output = f"All Customers ({len(results)} total):\n"
        for r in results:
            output += (
                f"\n  ID: {r[0]} | Name: {r[1]} | Room: {r[4]} | "
                f"Check-in: {r[5]} | Check-out: {r[6]}"
            )
        return output

In [40]:
def search_by_checkin_date(check_in_date):
    print(f"DATABASE TOOL CALLED: Searching by check-in date {check_in_date}", flush=True)
    with sqlite3.connect(DB) as conn:
        conn.execute("PRAGMA foreign_keys = ON")
        cursor = conn.cursor()

        cursor.execute('SELECT * FROM customers WHERE check_in_date = ?', (check_in_date,))
        results = cursor.fetchall()

        if not results:
            return f"No customers found with check-in date {check_in_date}."

        output = f"Customers checking in on {check_in_date} ({len(results)} total):\n"
        for r in results:
            output += (
                f"\n  ID: {r[0]} | Name: {r[1]} | Room: {r[4]} | "
                f"Check-out: {r[6]}"
            )
        return output

## Tools Def

In [41]:
get_customer_tool = {
        "name": "get_customer_details",
        "description": "Get details of a customer. MUST provide either customer_id OR name. Never call this without at least one of them.",
        "parameters": {
            "type": "object",
            "properties": {
                "identifier": {
                    "type": "string",
                    "description": "The unique id of the customer or the name of the customer",
                },
            },
            "required": [],
            "additionalProperties": False,
        }
}

In [42]:
save_customer_tool = {
    "name": "save_customer_details",
    "description": "Save a new customer booking. Requires name, phone, room number, check in date, check out date and Email.Special requests are optional.",
    "parameters": {
        "type": "object",
        "properties": {
            "name": {
                "type": "string",
                "description": "Full name of the customer."
            },
            "phone": {
                "type": "string",
                "description": "Phone number of the customer. 9 to 15 digits. May start with +."
            },
            "email": {
                "type": "string",
                "description": "Email address of the customer."
            },
            "room_number": {
                "type": "string",
                "description": "Room number to book. For example 101."
            },
            "check_in_date": {
                "type": "string",
                "description": "Check in date in YYYY-MM-DD format."
            },
            "check_out_date": {
                "type": "string",
                "description": "Check out date in YYYY-MM-DD format. Must be after check in date."
            },
            "special_requests": {
                "type": "string",
                "description": "Any special requests from the customer. Optional. For example late check-in."
            }
        },
        "required": ["name", "phone", "room_number", "check_in_date", "check_out_date","email"],
        "additionalProperties": False
    }
}

In [43]:
update_customer_tool = {
    "name": "update_customer_details",
    "description": "Update one or more details of an existing customer. Customer ID is required to find the customer. All other fields are optional — only provide the fields that need to change. Customer ID itself can never be changed.",
    "parameters": {
        "type": "object",
        "properties": {
            "customer_id": {
                "type": "string",
                "description": "The unique ID of the customer to update. Required."
            },
            "name": {
                "type": "string",
                "description": "New full name of the customer. Optional."
            },
            "phone": {
                "type": "string",
                "description": "New phone number. 9 to 15 digits. May start with +. Optional."
            },
            "email": {
                "type": "string",
                "description": "New email address. Optional."
            },
            "room_number": {
                "type": "string",
                "description": "New room number. Must exist and be available. Optional."
            },
            "check_in_date": {
                "type": "string",
                "description": "New check in date in YYYY-MM-DD format. Optional."
            },
            "check_out_date": {
                "type": "string",
                "description": "New check out date in YYYY-MM-DD format. Must be after check in date. Optional."
            },
            "special_requests": {
                "type": "string",
                "description": "New special requests. Pass empty string to clear existing requests. Optional."
            }
        },
        "required": ["customer_id"],
        "additionalProperties": False
    }
}

In [44]:
delete_customer_tool = {
    "name": "delete_customer",
    "description": "Delete a customer. Call first WITHOUT confirmation to show details and ask staff. Call again WITH confirmation=YES to delete or confirmation=NO to cancel.",
    "parameters": {
        "type": "object",
        "properties": {
            "customer_id": {
                "type": "string",
                "description": "The unique ID of the customer to delete."
            },
            "confirmation": {
                "type": "string",
                "description": "Staff confirmation. Omit on first call. Pass YES to delete or NO to cancel on second call."
            }
        },
        "required": ["customer_id"],
        "additionalProperties": False
    }
}

In [45]:
check_rooms_tool = {
    "name": "check_rooms",
    "description": "Check all room numbers and their availability. Shows which rooms are available and which are occupied.",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False
    }
}

In [46]:
list_customers_tool = {
    "name": "list_all_customers",
    "description": "List all registered customers. Use when staff asks to see all customers or get a full list.",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False
    }
}

In [47]:
search_checkin_tool = {
    "name": "search_by_checkin_date",
    "description": "Find all customers checking in on a specific date. Date must be in YYYY-MM-DD format.",
    "parameters": {
        "type": "object",
        "properties": {
            "check_in_date": {
                "type": "string",
                "description": "The check-in date to search for in YYYY-MM-DD format."
            }
        },
        "required": ["check_in_date"],
        "additionalProperties": False
    }
}

In [48]:
# tools

tools = [
    {"type": "function", "function": get_customer_tool},
    {"type": "function", "function": save_customer_tool},
    {"type": "function", "function": update_customer_tool},
    {"type": "function", "function": delete_customer_tool},
    {"type": "function", "function": check_rooms_tool},
    {"type": "function", "function": list_customers_tool},
    {"type": "function", "function": search_checkin_tool}
]


In [49]:
tools

[{'type': 'function',
  'function': {'name': 'get_customer_details',
   'description': 'Get details of a customer. MUST provide either customer_id OR name. Never call this without at least one of them.',
   'parameters': {'type': 'object',
    'properties': {'identifier': {'type': 'string',
      'description': 'The unique id of the customer or the name of the customer'}},
    'required': [],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'save_customer_details',
   'description': 'Save a new customer booking. Requires name, phone, room number, check in date, check out date and Email.Special requests are optional.',
   'parameters': {'type': 'object',
    'properties': {'name': {'type': 'string',
      'description': 'Full name of the customer.'},
     'phone': {'type': 'string',
      'description': 'Phone number of the customer. 9 to 15 digits. May start with +.'},
     'email': {'type': 'string',
      'description': 'Email address of the customer

## Chat Function

In [50]:
def chat(message, history):

    chat_history = []

    for h in history:
        chat_history.append({"role":h["role"], "content":h["content"]})

    messages = [{"role":"system", "content":system_prompt}] + chat_history + [{"role":"user", "content":message}]
    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools
    )

    while response.choices[0].finish_reason == 'tool_calls':
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.extend(response)

        response = openai.chat.completions.create(model=MODEL,messages=messages, tools=tools)
    return response.choices[0].message.content

In [51]:
def handle_tool_call(message):
    response = []
    for tool_call in message.tool_calls:
        arguments = json.loads(tool_call.function.arguments)
        result = None

        if tool_call.function.name == "get_customer_details":
            result = get_customer_details(
                identifier = arguments.get("identifier")
            )

        elif tool_call.function.name == "save_customer_details":
            result = save_customer_details(
                name             = arguments.get("name"),
                phone            = arguments.get("phone"),
                email            = arguments.get("email"),
                room_number      = arguments.get("room_number"),
                check_in_date    = arguments.get("check_in_date"),
                check_out_date   = arguments.get("check_out_date"),
                special_requests = arguments.get("special_requests")
            )

        elif tool_call.function.name == "update_customer_details":
            result = update_customer_details(
                customer_id      = arguments.get("customer_id"),
                name             = arguments.get("name",             UNCHANGED),
                phone            = arguments.get("phone",            UNCHANGED),
                email            = arguments.get("email",            UNCHANGED),
                room_number      = arguments.get("room_number",      UNCHANGED),
                check_in_date    = arguments.get("check_in_date",    UNCHANGED),
                check_out_date   = arguments.get("check_out_date",   UNCHANGED),
                special_requests = arguments.get("special_requests", UNCHANGED)
            )

        elif tool_call.function.name == "delete_customer":
            result = delete_customer(
                customer_id  = arguments.get("customer_id"),
                confirmation = arguments.get("confirmation")
            )

        elif tool_call.function.name == "check_rooms":
            result = check_rooms()
            
        elif tool_call.function.name == "list_all_customers":
            result = list_all_customers()

        elif tool_call.function.name == "search_by_checkin_date":
            result = search_by_checkin_date(
                check_in_date = arguments.get("check_in_date")
            )

        else:
            result = "Unknown tool called."

        response.append({
            "role"        : "tool",
            "content"     : str(result),
            "tool_call_id": tool_call.id
        })

    return response

In [52]:
gradio.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "d:\NoobDev\AI\Hotel Management AI Agent\.venv\Lib\site-packages\httpx\_transports\default.py", line 101, in map_httpcore_exceptions
    yield
  File "d:\NoobDev\AI\Hotel Management AI Agent\.venv\Lib\site-packages\httpx\_transports\default.py", line 250, in handle_request
    resp = self._pool.handle_request(req)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\NoobDev\AI\Hotel Management AI Agent\.venv\Lib\site-packages\httpcore\_sync\connection_pool.py", line 256, in handle_request
    raise exc from None
  File "d:\NoobDev\AI\Hotel Management AI Agent\.venv\Lib\site-packages\httpcore\_sync\connection_pool.py", line 236, in handle_request
    response = connection.handle_request(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\NoobDev\AI\Hotel Management AI Agent\.venv\Lib\site-packages\httpcore\_sync\connection.py", line 101, in handle_request
    raise exc
  File "d:\NoobDev\AI\Hotel Management AI Agent\.venv\Lib\site-packages\htt